In [ ]:
import pylbo as pb
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
case = "HEL1"

In [ ]:
os.system("rm "+case+"/output/*")
os.system("rm "+case+"/parfiles/*")

In [ ]:
config = {
    "number_of_runs": 50,
    "gridpoints": 101,
    "equilibrium_type": "user_defined",
    "parameters": {

        "V": 1.63,
        "cte_rho0": 1.0,
        "cte_p0": 1.0,
        "Bz0": 0.25,
        "rj": 1.0,
        "rc": 0.5 if case=="HEL2" else 2.0 ,
        "k2": -1.0,
        "k3": np.linspace(0.5, 8.0 , 50)
    },
    "write_eigenfunctions": True,
    "basename_datfile": "khcd_spectrum_"+case,
    "write_derived_eigenfunctions": True,
    "output_folder": case+"/output",
    "solver": "QR-cholesky",
    "use_defaults": False
}

os.system("rm "+case+"/output/*")
os.system("rm "+case+"/parfiles/*")

parfiles = pb.generate_parfiles(config, basename="khcd_spectrum_"+case, output_dir=case)

In [ ]:
pb.run_legolas(parfiles, executable="legolas", nb_cpus=8)

In [ ]:
datfiles = sorted(Path(case+"/output/").glob("*khcd_spectrum_"+case+"*.dat"))

series = pb.load_series(datfiles)

In [ ]:
k3 = []
growthRate = []
frequencies = []
eigenfunctions = []

for spectra in series:

    eigenVal = spectra.eigenvalues

    eigenPos = []

    for eigenId, evs in enumerate(eigenVal):
        
        omega = evs.real
        gamma = evs.imag

        if(gamma>1e-2):
            growthRate.append(gamma)
            frequencies.append(omega)

            k3.append(spectra.parameters["k3"])

    eigenPos.append(eigenId)

    efs = spectra.get_eigenfunctions(ev_idxs=eigenPos)

    eigenfunctions.append(efs)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(k3, growthRate, s=4, color="gray",  label="all modes", alpha=0.5)

ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("$k_3$")
ax.set_ylabel(r"Growth rate ")
ax.set_title("KH vs CD mode separation")
ax.legend()
plt.tight_layout()
plt.show()